# Logistics Shipment Dataset 

## Feature Engineering

### This notebook focuses on creating analytical features to evaluate logistics performance, cost efficiency, and delivery reliability.

### The features created in this stage will support downstream analysis and dashboard development.

In [6]:
import pandas as pd

df = pd.read_csv("../data/processed/shipments_clean.csv")

date_columns = [
    "PQ First Sent to Client Date",
    "PO Sent to Vendor Date",
    "Scheduled Delivery Date",
    "Delivered to Client Date",
    "Delivery Recorded Date",
    "pq_sent_to_client_date",
    "po_sent_to_vendor_date"
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10324 entries, 0 to 10323
Data columns (total 43 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   ID                            10324 non-null  int64         
 1   Project Code                  10324 non-null  str           
 2   PQ #                          10324 non-null  str           
 3   PO / SO #                     10324 non-null  str           
 4   ASN/DN #                      10324 non-null  str           
 5   Country                       10324 non-null  str           
 6   Managed By                    10324 non-null  str           
 7   Fulfill Via                   10324 non-null  str           
 8   Vendor INCO Term              10324 non-null  str           
 9   Shipment Mode                 9964 non-null   str           
 10  PQ First Sent to Client Date  7643 non-null   datetime64[us]
 11  PO Sent to Vendor Date        4592 non-

C:\Users\danie\AppData\Local\Temp\ipykernel_14280\2566775687.py:16: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce")
C:\Users\danie\AppData\Local\Temp\ipykernel_14280\2566775687.py:16: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce")


In [ ]:
df_cost = df.dropna(subset=["weight_kg", "freight_cost_usd"])


import numpy as np

# Calculate cost per kg, handling division by zero and missing values
df["cost_per_kg"] = df["freight_cost_usd"] / df["weight_kg"].replace(0, np.nan)


#df["delivery_delay_days"] = (
#    df["Delivered to Client Date"] -
#    df["Scheduled Delivery Date"]
#).dt.days

#df[["ID",
#     "Delivered to Client Date", "Scheduled Delivery Date", "delivery_delay_days", 
#     "freight_cost_usd", "weight_kg", "cost_per_kg"]].iloc[20:25]


df_cost = df_cost[df_cost["weight_kg"] > 1]


q1 = df_cost["cost_per_kg"].quantile(0.25)
q3 = df_cost["cost_per_kg"].quantile(0.75)
iqr = q3 - q1

df_cost = df_cost[
    (df_cost["cost_per_kg"] >= q1 - 1.5 * iqr) &
    (df_cost["cost_per_kg"] <= q3 + 1.5 * iqr)
]


df_cost["shipment_size"] = pd.qcut(
    df_cost["weight_kg"],
    q=4,
    labels=["Small", "Medium", "Large", "Very Large"]
)

df_cost["cost_per_kg_percentile"] = df_cost["cost_per_kg"].rank(pct=True)

df_cost["route"] = df_cost["Manufacturing Site"] + " → " + df_cost["Country"]


df_time = df.dropna(subset=["po_sent_to_vendor_date", "Delivered to Client Date"])



df[[
    "cost_per_kg",
    "delivery_delay_days",
    "order_to_delivery_days",
    "delivery_status",
    "route",
    "shipment_size",
    "cost_per_kg_percentile"
]].describe()


,cost_per_kg,delivery_delay_days,order_to_delivery_days,cost_per_kg_percentile
count,2592.000000,2592.000000,2592.000000,2592.000000
mean,10.305804,0.881559,118.795139,0.500193
std,7.435764,11.316380,70.342949,0.288731
min,0.007576,-169.000000,12.000000,0.000386
25%,4.344579,0.000000,67.000000,0.250289
50%,8.917983,0.000000,105.000000,0.500193
75%,14.816007,0.000000,153.000000,0.750096
max,32.320000,150.000000,616.000000,1.000000


In [ ]:
df_time = df.dropna(
    subset=["po_sent_to_vendor_date", "Delivered to Client Date"]
).copy()


df_time["order_to_delivery_days"] = (
    df_time["Delivered to Client Date"] - 
    df_time["po_sent_to_vendor_date"]
).dt.days

df_time = df_time[df_time["order_to_delivery_days"] >= 0]


df_time["delivery_status"] = pd.cut(
    df_time["delivery_delay_days"],
    bins=[-999, 0, 5, 999],
    labels=["On Time", "Slight Delay", "Severe Delay"]
)



Cost and Efficiency Metrics

To evaluate logistics efficiency, the following features were created:

- cost_per_kg: measures transportation cost efficiency  
- delivery_delay_days: measures deviation from scheduled delivery  
- order_to_delivery_days: total lead time  

These metrics allow identifying inefficiencies across routes,
vendors, and shipment sizes.

Process Flow Considerations

Not all shipments follow the same operational flow.

Some records contain process indicators such as "Pre-PQ Process"
or "N/A - From RDC", meaning that certain steps (e.g. PQ or PO)
were not applicable.

Therefore:

• Date-based metrics were calculated only for applicable records  
• Process types were analyzed separately to avoid biased results  

This approach ensures accurate interpretation of logistics performance.

In [ ]:
df.to_csv("../data/processed/shipments_featured.csv", index=False)